# Arabic PDF to Searchable PDF Converter (1251 Pages Layout-Preserving OCR)

This notebook converts scanned or image-based Arabic PDFs (such as multi-page dictionaries or books) into searchable PDFs with an invisible OCR text layer while strictly preserving the original visual layout formatting.

### Features:
- Supports **Google Colab GPU (T4)** or **Kaggle** execution.
- Invisible text overlay (`render_mode=3`) at exact OCR bounding boxes.
- Arabic shaping (`arabic_reshaper` + `python-bidi`) for accurate text search.
- Checkpoint resume capability: continue from page 498/1251 seamlessly!

In [ ]:
# Step 1: Install required dependencies
import os
import sys
os.system(f"{sys.executable} -m pip install -q pymupdf easyocr opencv-python-headless arabic-reshaper python-bidi")

In [ ]:
# Step 2: Set input & output paths
import os
import urllib.request

if os.path.exists("/kaggle/working"):
    BASE_DIR = "/kaggle/working"
else:
    BASE_DIR = "."

INPUT_PDF = os.path.join(BASE_DIR, "input_book.pdf")
OUTPUT_PDF = os.path.join(BASE_DIR, "searchable_book.pdf")
CHECKPOINT_FILE = os.path.join(BASE_DIR, "checkpoint.json")

# Google Drive source PDF
GDRIVE_PDF_ID = "1qyPmkzNgvyyJrWk2NMBD4w_aE6i9TIuG"
DOWNLOAD_URL = f"https://drive.google.com/uc?export=download&id={GDRIVE_PDF_ID}"

if not os.path.exists(INPUT_PDF):
    print(f"Downloading source PDF to {INPUT_PDF}...")
    try:
        urllib.request.urlretrieve(DOWNLOAD_URL, INPUT_PDF)
        print(f"Downloaded successfully ({os.path.getsize(INPUT_PDF)} bytes).")
    except Exception as e:
        print(f"Download error: {e}.")

In [ ]:
# Step 3: Define layout-preserving searchable PDF engine
import sys
import json
import tempfile
from pathlib import Path
import pymupdf
import easyocr
import cv2
import numpy as np
import arabic_reshaper
from bidi.algorithm import get_display
import torch

FONT_CANDIDATES = [
    "/usr/share/fonts/opentype/unifont/unifont.otf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    "/usr/share/fonts/opentype/ipafont-gothic/ipag.ttf",
]

def find_unicode_font(custom_path=None):
    if custom_path and os.path.exists(custom_path):
        return custom_path
    for font_path in FONT_CANDIDATES:
        if os.path.exists(font_path):
            return font_path
    for root, _, files in os.walk("/usr/share/fonts"):
        for file in files:
            if file.endswith((".ttf", ".otf")):
                return os.path.join(root, file)
    raise FileNotFoundError("No Unicode font found for PDF embedding.")

def prepare_arabic_for_pdf(text):
    if not text:
        return ""
    try:
        reshaped = arabic_reshaper.reshape(text)
        return get_display(reshaped)
    except Exception:
        return text

def save_checkpoint(checkpoint_path, last_page, pdf_path, output_path):
    checkpoint_data = {
        "last_processed_page": last_page,
        "input_pdf": pdf_path,
        "output_pdf": output_path
    }
    with open(checkpoint_path, "w", encoding="utf-8") as f:
        json.dump(checkpoint_data, f, indent=2)

def load_checkpoint(checkpoint_path):
    if os.path.exists(checkpoint_path):
        try:
            with open(checkpoint_path, "r", encoding="utf-8") as f:
                return json.load(f).get("last_processed_page", 0)
        except Exception:
            pass
    return 0

def convert_pdf_to_searchable(
    input_pdf,
    output_pdf,
    start_page=1,
    end_page=None,
    dpi=150,
    use_gpu=None,
    resume=True,
    checkpoint_file=CHECKPOINT_FILE,
    save_interval=10
):
    if not os.path.exists(input_pdf):
        raise FileNotFoundError(f"Input file not found: {input_pdf}")

    if use_gpu is None:
        use_gpu = torch.cuda.is_available()

    cp_path = Path(checkpoint_file)
    active_font_path = find_unicode_font()
    print(f"Using font: {active_font_path}")

    src_doc = pymupdf.open(input_pdf)
    total_pages = len(src_doc)
    if end_page is None or end_page > total_pages:
        end_page = total_pages

    actual_start_page = start_page
    if resume:
        last_done = load_checkpoint(cp_path)
        if last_done >= end_page:
            print(f"🎉 Conversion already finished up to page {last_done}/{total_pages}!")
            src_doc.close()
            return
        elif last_done > 0:
            actual_start_page = max(start_page, last_done + 1)
            print(f"🔄 Resuming from checkpoint page {actual_start_page}...")

    print(f"Initializing EasyOCR (GPU={use_gpu})...")
    reader = easyocr.Reader(['ar'], gpu=use_gpu)

    out_doc = pymupdf.open()
    if os.path.exists(output_pdf) and actual_start_page > 1:
        try:
            existing_doc = pymupdf.open(output_pdf)
            out_doc.insert_pdf(existing_doc)
            existing_doc.close()
            print(f"Loaded existing output PDF ({len(out_doc)} pages).")
        except Exception as e:
            print(f"Starting fresh output document ({e}).")
            out_doc = pymupdf.open()

    font_bytes = open(active_font_path, "rb").read()
    print(f"Processing pages {actual_start_page} to {end_page} of {total_pages}...")

    for page_idx in range(actual_start_page - 1, end_page):
        page_num = page_idx + 1
        page = src_doc[page_idx]

        new_page = out_doc.new_page(width=page.rect.width, height=page.rect.height)
        new_page.show_pdf_page(page.rect, src_doc, page_idx)

        pix = page.get_pixmap(dpi=dpi)
        pix_bytes = pix.tobytes("png")
        nparr = np.frombuffer(pix_bytes, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        enhanced = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)

        with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp_file:
            tmp_path = tmp_file.name
            cv2.imwrite(tmp_path, enhanced)

        try:
            results = reader.readtext(tmp_path, detail=1, paragraph=False)
        finally:
            if os.path.exists(tmp_path):
                os.remove(tmp_path)

        scale_x = page.rect.width / pix.width
        scale_y = page.rect.height / pix.height
        new_page.insert_font(fontname="ufont", fontbuffer=font_bytes)

        for bbox, text, prob in results:
            text = text.strip()
            if not text or prob < 0.15:
                continue

            x_coords = [p[0] * scale_x for p in bbox]
            y_coords = [p[1] * scale_y for p in bbox]
            x0, x1 = max(0, min(x_coords)), min(page.rect.width, max(x_coords))
            y0, y1 = max(0, min(y_coords)), min(page.rect.height, max(y_coords))

            rect = pymupdf.Rect(x0, y0, x1, y1)
            formatted_text = prepare_arabic_for_pdf(text)
            fontsize = max(6, (y1 - y0) * 0.75)

            try:
                new_page.insert_textbox(
                    rect, formatted_text, fontname="ufont", fontsize=fontsize, render_mode=3
                )
            except Exception:
                try:
                    new_page.insert_text(
                        (x0, y1), formatted_text, fontname="ufont", fontsize=fontsize, render_mode=3
                    )
                except Exception:
                    pass

        save_checkpoint(cp_path, page_num, input_pdf, output_pdf)
        print(f"Page {page_num}/{end_page} processed.")

        if (page_num % save_interval == 0) or (page_num == end_page):
            out_doc.save(output_pdf, incremental=False)
            print(f" Checkpoint & PDF saved at page {page_num}/{end_page}.")

    out_doc.close()
    src_doc.close()
    print(f"\n Searchable PDF successfully generated at {output_pdf}")

In [ ]:
# Step 4: Run document conversion starting from page 498 (or start_page=1 if fresh run)
convert_pdf_to_searchable(
    input_pdf=INPUT_PDF,
    output_pdf=OUTPUT_PDF,
    start_page=498, # Start directly from page 498 to finish the remaining pages!
    end_page=1251,
    dpi=150,
    use_gpu=None,  # Auto GPU if available
    resume=True,
    save_interval=10
)

In [ ]:
# Step 5: Provide download link for generated searchable PDF file
from IPython.display import FileLink, display
import os

if os.path.exists(OUTPUT_PDF):
    size_mb = os.path.getsize(OUTPUT_PDF) / (1024 * 1024)
    print(f"🎉 Searchable PDF is ready! File size: {size_mb:.2f} MB")
    display(FileLink(OUTPUT_PDF))
else:
    print("Output PDF not found. Make sure conversion finished successfully.")